In [1]:
from pathlib import Path
import sys

import simcentralconnect
from System import Array, String, Double, AggregateException, InvalidOperationException
import os

In [2]:
# Helper function to attempt to open a simulation and gracefully handle the case when it doesn't exist.
def open_simulation(sm, simName):
    opened = False
    try:
        opened = sm.OpenSimulation(simName).Result
    except AggregateException as ex:
        if not isinstance(ex.InnerException, InvalidOperationException) \
            or "simulation doesn't exists" not in ex.InnerException.Message:
            raise
    return opened

In [3]:
# Get access to various managers used in this script
sc = simcentralconnect.connect().Result
sm = sc.GetService("ISimulationManager")
mm = sc.GetService("IModelManager")
cm = sc.GetService("IConnectorManager")
dm = sc.GetService("IDiagramManager")
vm = sc.GetService("IVariableManager")

SimCentral.Client.Auto loaded from: C:\Program Files\SimSci\SimCentral\Scripting\DotNetFwk


In [ ]:
SimNameScriptingEx = "Scripting_Ex_1"


#os.chdir(Path(__file__).parent.absolute())


# Example of how to set options.  API logging is ON by default.  Disabled by changing to 'false'
sc.SetOptions(repr({'Timeout': 40000, 'EnableApiLogging': 'true'}))

# Attempt to open the example simulation (it may exist from a previous run)
opened = open_simulation(sm, SimNameScriptingEx);
if opened:
    # If the simulation was opened, delete all the connectors and models

    connectors = dm.GetConnectors(SimNameScriptingEx).Result
    for connector in connectors:
        name = connector["name"].ToString()
        cm.RemoveConnector(SimNameScriptingEx, name).Result
       
        
    models = dm.GetModels(SimNameScriptingEx).Result
    for model in models:
        name = model["name"].ToString()
        mm.RemoveModel(SimNameScriptingEx, name).Result
       
        
else:
    # If the simulation doesn't exist, create it.
    simName = sm.CreateSim(os.getlogin(), None).Result
    sm.RenameSim(simName, SimNameScriptingEx).Result
    



In [5]:
# Create a source and sink, and connect them.
srcName = mm.AddModel(SimNameScriptingEx, None, "Lib:SteamLib.Source", 100, 100).Result
pumpName = mm.AddModel(SimNameScriptingEx, None, "Lib:SteamLib.Pump", 200, 100).Result
valName = mm.AddModel(SimNameScriptingEx, None, "Lib:SteamLib.Valve", 300, 125).Result
conName = mm.AddModel(SimNameScriptingEx, None, "Lib:Controls.PID", 300, 25).Result
transName = mm.AddModel(SimNameScriptingEx, None, "Lib:Controls.Trans", 400,75).Result
snkName = mm.AddModel(SimNameScriptingEx, None, "Lib:SteamLib.Sink", 500, 90).Result

cm.AddConnector(SimNameScriptingEx, None, srcName + ".Out", pumpName + ".In").Result
cm.AddConnector(SimNameScriptingEx, None, pumpName + ".Out", valName + ".In").Result
cm.AddConnector(SimNameScriptingEx, None, valName + ".Out", transName + ".In").Result
cm.AddConnector(SimNameScriptingEx, None, transName + ".Out", snkName + ".In").Result


# Set control type
cont, trant = vm.GetVariableValue(SimNameScriptingEx, Array[String]([ transName + ".Type" , conName + ".Type"])).Result   
#vm.SetVariableValue(SimNameScriptingEx, Array[String]([ transName + ".Type" , conName + ".Type"]),  Array[String](["MassFlow", "MassFlow"])).Result    


# Add connectors for controls
cm.AddConnector(SimNameScriptingEx, None, conName + ".Output", valName + ".Control").Result
cm.AddConnector(SimNameScriptingEx, None, transName + ".Trans", conName + ".Input").Result

# Specify required values by reading the current values and writing them back
srcw, srct, srcP = vm.GetVariableValue(SimNameScriptingEx, Array[String]([srcName + ".W", srcName + ".T", srcName + ".P"])).Result   
print(f"Scripting: .W: {srcw}, .T: {srct}, .P: {srcP}, ")

vm.SetVariableValue(SimNameScriptingEx, Array[String]([srcName + ".W", srcName + ".T", srcName + ".P"]), Array[Double]([srcw, srct, srcP]), Array[String](["kg/s", "K", "kPa"])).Result    

# Confirm that the simulation has all required data, is property specified and solved
ss = sm.GetSimulationStatus(SimNameScriptingEx).Result
if not ss[0] or not ss[1] or not ss[2]:
    print("Scripting: *** Invalid simulation status:")
    print(f"Scripting: {ss[0]}, {ss[1]}, {ss[2]}")



Scripting: .W: 6.0, .T: 644.2611111111112, .P: 6894.76, 
Scripting: *** Invalid simulation status:
Scripting: False, True, True


In [ ]:
# Close all Open Simulations
sm.CloseOpenSimulations().Result

In [ ]:
# Close all Open Simulations
sm.CloseOpenSimulations().Result